In [2]:
import csv
import openpyxl
import os

file_name = 'DataCoSupplyChainDataset.csv'

if not os.path.exists(file_name):
    print(f"❌ שגיאה: הקובץ '{file_name}' לא נמצא בתיקייה!")
else:
    print("⏳ מתחיל לקרוא ולנתח את הנתונים (זה עשוי לקחת כמה שניות, בכל זאת 180 אלף שורות)...")
    
    # --- הגדרת מילונים (Dictionaries) לשמירת התוצאות שלנו ---
    # מילון לשמירת נתוני הרווח לפי סטטוס המשלוח
    profit_by_delivery = {} 
    
    # מילון לספירת מקרי ההונאה לפי קטגוריית מוצר
    fraud_categories = {}
    
    # מילון לחישוב איחורים לפי אזור גיאוגרפי
    region_delays = {}
    
    row_count = 0
    
    # --- שלב 1: פתיחת הקובץ וקריאה שורה אחרי שורה (חוסך זיכרון למחשב!) ---
    with open(file_name, mode='r', encoding='ISO-8859-1') as file:
        reader = csv.DictReader(file) # קורא את הקובץ ומתייחס לכל שורה כמו מילון
        
        for row in reader:
            row_count += 1
            
            # חילוץ הנתונים מהשורה הנוכחית
            # אנחנו משתמשים ב-float ו-int כדי להפוך את הטקסט למספרים
            try:
                profit = float(row['Order Profit Per Order'])
                real_shipping_days = int(row['Days for shipping (real)'])
                scheduled_shipping_days = int(row['Days for shipment (scheduled)'])
            except ValueError:
                continue # אם חסר מספר בשורה הזו, נדלג עליה ונמשיך הלאה
                
            delivery_status = row['Delivery Status']
            order_status = row['Order Status']
            category_name = row['Category Name']
            region = row['Order Region']
            
            # --- שלב 2: ניתוח הנתונים (הכנסה למילונים) ---
            
            # א. חישוב רווח לפי סטטוס משלוח
            if delivery_status not in profit_by_delivery:
                profit_by_delivery[delivery_status] = {'total_profit': 0, 'count': 0}
            
            profit_by_delivery[delivery_status]['total_profit'] += profit
            profit_by_delivery[delivery_status]['count'] += 1
            
            # ב. ספירת הונאות (אם הסטטוס הוא הונאה, נוסיף 1 לקטגוריה)
            if order_status == 'SUSPECTED_FRAUD':
                if category_name not in fraud_categories:
                    fraud_categories[category_name] = 0
                fraud_categories[category_name] += 1
                
            # ג. חישוב זמן האיחור (ימים בפועל פחות ימים מתוכננים)
            delay = real_shipping_days - scheduled_shipping_days
            if region not in region_delays:
                region_delays[region] = {'total_delay': 0, 'count': 0}
            
            region_delays[region]['total_delay'] += delay
            region_delays[region]['count'] += 1

    print(f"✅ סיימנו בהצלחה לעבור על {row_count} שורות!\n")

    # --- שלב 3: מציאת ה"טופ 5" (החלק המעניין) ---
    
    # מיון מילון ההונאות כדי למצוא את 5 הקטגוריות הכי בעייתיות
    # sorted מסדר את המילון מהגדול לקטן לפי הערך (x[1])
    top_5_frauds = sorted(fraud_categories.items(), key=lambda x: x[1], reverse=True)[:5]
    
    # חישוב הממוצע של איחורים לכל אזור, ומיון ל-5 הכי גרועים
    region_avg_delays = []
    for reg, data in region_delays.items():
        avg = data['total_delay'] / data['count']
        region_avg_delays.append((reg, avg))
    
    top_5_delayed_regions = sorted(region_avg_delays, key=lambda x: x[1], reverse=True)[:5]

    # --- שלב 4: ייצוא לאקסל בעזרת openpyxl (כמו בפרויקט הראשון!) ---
    report_name = "Supply_Chain_Report_Basic.xlsx"
    wb = openpyxl.Workbook()
    
    # גיליון 1: רווחיות
    ws1 = wb.active
    ws1.title = "Profit by Delivery"
    ws1.append(["Delivery Status", "Average Profit ($)", "Total Orders", "Total Profit ($)"]) # כותרות
    for status, data in profit_by_delivery.items():
        avg_profit = data['total_profit'] / data['count']
        ws1.append([status, round(avg_profit, 2), data['count'], round(data['total_profit'], 2)])
        
    # גיליון 2: הונאות
    ws2 = wb.create_sheet("Top Frauds")
    ws2.append(["Category Name", "Number of Frauds"])
    for cat, count in top_5_frauds:
        ws2.append([cat, count])
        
    # גיליון 3: איחורים לפי אזור
    ws3 = wb.create_sheet("Regional Delays")
    ws3.append(["Region", "Average Delay (Days)"])
    for reg, avg in top_5_delayed_regions:
        ws3.append([reg, round(avg, 2)])

    wb.save(report_name)
    
    print("📋 הצצה לנתוני ההונאה (Top 5):")
    for item in top_5_frauds:
        print(f"- {item[0]}: {item[1]} cases")
        
    print(f"\n🚀 הפרויקט הסתיים! הדוח נשמר בקובץ: {report_name}")

⏳ מתחיל לקרוא ולנתח את הנתונים (זה עשוי לקחת כמה שניות, בכל זאת 180 אלף שורות)...
✅ סיימנו בהצלחה לעבור על 180519 שורות!

📋 הצצה לנתוני ההונאה (Top 5):
- Cleats: 560 cases
- Men's Footwear: 516 cases
- Women's Apparel: 481 cases
- Indoor/Outdoor Games: 439 cases
- Fishing: 394 cases

🚀 הפרויקט הסתיים! הדוח נשמר בקובץ: Supply_Chain_Report_Basic.xlsx
